# Fixed against variable land use

Holding land use still is usually defended as the cautious choice. A forecast
that lets employment move might flatter the scheme; a forecast that holds it
still cannot, so the fixed number is the one that is safe to put in front of a
committee.

This notebook runs both cases in one execution and prints them beside each other.
Same scheme file, same cost matrix, same 14,863 jobs of growth, same balancing
tolerance. In one case the growth is shared out as employment already sits, and
in the other it follows the accessibility the scheme created, at the elasticity
you set, for as many iterations as you allow.

What you are testing is whether "conservative" names a direction. If it does,
the fixed forecast should sit on one side of the variable one, consistently,
and you should be able to say which side before you run anything. Two of the
figures below disagree with each other about that. Find them.

Work through the sections in order, from the top of the page to the bottom.

## Where the files are

JupyterLite runs inside your browser. Nothing is installed on your machine and
you do not need administrator rights, which is why this page opens on a
locked-down work machine.

The notebook and its data arrived with the site, so there is nothing to download
and nothing to upload. Open the file browser - the panel down the left-hand
side, or the folder icon in the far-left sidebar if it is not showing - and you
will find this arrangement already in place:

```
fixed-vs-variable/
    fixed-vs-variable.ipynb
    data/
        zones_msoa.csv
        trip_ends_msoa.csv
        cost_matrix_msoa.csv
        scheme_cost_adjustments_narrow.csv
```

Every path in the code below assumes it. The notebook sits at the top of the
folder and the data sits one level under it, so moving either one will break the
loading section.

**What happens to anything you change**

Because there is no server behind this, whatever you save goes into your
browser's own storage rather than onto a network drive. That has two
consequences worth taking seriously. Anything you want to keep should be
downloaded - right-click the file in the file browser and choose **Download**.
And if you clear your browsing data, or if your employer's IT policy clears it
for you, your saved work goes with it.

Do not edit the CSV files. If you want to try something out on them, duplicate
one first and work on the copy.

**Getting back to the original**

Should you change the notebook and want the version you started with, use
**Help > Clear Browser Data**. But read the warning it gives you before
confirming. It removes everything you have stored on this site, for every
notebook here, and it cannot be undone, so download anything you care about
first.

## Checking the files are where you think they are

Run the cell below first. It reports what it can see, which is faster than
reading an error message later and guessing at the cause.

In [ ]:
import os

DATA_FOLDER = "data"

expected = [
    "zones_msoa.csv",
    "trip_ends_msoa.csv",
    "cost_matrix_msoa.csv",
    "scheme_cost_adjustments_narrow.csv",
]

print("Looking in:", os.path.abspath(DATA_FOLDER))
print()

if not os.path.isdir(DATA_FOLDER):
    print("That folder does not exist yet.")
    print("Check the folder names and check where this notebook is saved.")
else:
    found = sorted(os.listdir(DATA_FOLDER))
    for name in expected:
        status = "found" if name in found else "MISSING"
        print(f"  {name:38s} {status}")

## Parameters

Two values, and the only cell in this notebook you will change.

ACCESS_ELASTICITY is the parameter you set when you wrote the reallocation rule,
where it appeared as epsilon. It carries no units and the name has not changed
since.

LOOP_ITERATIONS is the number of times land use and transport are allowed round
the loop before the model stops. Because the reading on convergence already
established what the sequence does at this scale, the default is set on that
evidence rather than on principle: the first iteration carries almost all of the
movement, the second shifts the largest zone by about one job, and the third
takes a hundredth of a job back out again.

Raising it above three leaves the comparison unchanged at the elasticity set
beside it, which is the loop reporting that it has converged rather than the
notebook disregarding you. The trajectory table below is where a change to this
parameter registers.

Everything else is held. Beta stays at 0.1185 per minute, the opportunity
variable at jobs, the scheme definition at narrow, and the growth at 14,863
jobs. Both cases read those same fixed values, so anything separating the two
forecasts came from the two parameters below and from nothing else.

Run at 1.0 and 3 first. Then change one of them, restart, and run everything
again from the top.

In [ ]:
# ---------------------------------------------------------------------------
# PARAMETERS
# ---------------------------------------------------------------------------

ACCESS_ELASTICITY = 1.0    # how strongly jobs follow accessibility, unitless

LOOP_ITERATIONS = 3        # times land use and transport go round the loop
                           # in the variable case. The fixed case never goes
                           # round it at all.

# ---------------------------------------------------------------------------

## Loading the data, and the scheme

Three data files and the scheme file. The cost matrix is much the largest at
21,025 rows, so give the cell a moment before deciding it has stalled.

The narrow scheme file is applied exactly as it was in the scheme test, 42 rows
of the matrix rewritten and nothing else touched. It links the seven Washington
MSOAs to the three zones holding the regional centres, in both directions, and
the cell below works out which zones are which from the file rather than from a
list typed into the notebook.

In [ ]:
import numpy as np
import pandas as pd

BETA = 0.1185                    # fixed, per minute of generalised cost
OPPORTUNITY = "jobs"             # fixed
SCHEME_DEFINITION = "narrow"     # fixed
GROWTH_TOTAL = 14863             # jobs to be allocated, fixed
TOLERANCE = 0.01                 # balancing tolerance, in trips
MAX_ITERATIONS = 100             # ceiling on the balancing, not on the loop

zones = pd.read_csv(f"{DATA_FOLDER}/zones_msoa.csv", encoding="utf-8-sig")
trip_ends = pd.read_csv(f"{DATA_FOLDER}/trip_ends_msoa.csv",
                        encoding="utf-8-sig")
costs = pd.read_csv(f"{DATA_FOLDER}/cost_matrix_msoa.csv",
                    encoding="utf-8-sig")

zone_ids = list(zones["zone_id"])
position = {zone: i for i, zone in enumerate(zone_ids)}

details = (zones[["zone_id", "zone_name", "local_authority"]]
           .merge(trip_ends[["zone_id", "resident_workers", "jobs"]],
                  on="zone_id")
           .set_index("zone_id")
           .reindex(zone_ids))

zone_names = list(details["zone_name"])
authorities = list(details["local_authority"])
base_jobs = details[OPPORTUNITY].values.astype(float)
base_workers = details["resident_workers"].values.astype(float)

base_cost = (costs
             .pivot(index="origin_id", columns="destination_id",
                    values="gc_min")
             .reindex(index=zone_ids, columns=zone_ids)
             .values.astype(float))

distance = (costs
            .pivot(index="origin_id", columns="destination_id",
                   values="distance_km")
            .reindex(index=zone_ids, columns=zone_ids)
            .values.astype(float))

adjustments = pd.read_csv(
    f"{DATA_FOLDER}/scheme_cost_adjustments_{SCHEME_DEFINITION}.csv",
    encoding="utf-8-sig", comment="#")

scheme_cost = base_cost.copy()
for origin, destination, change in adjustments.itertuples(index=False):
    i, j = position[origin], position[destination]
    scheme_cost[i, j] = base_cost[i, j] + change

# Each Washington zone is listed against three centres and each centre against
# seven Washington zones, so the row counts separate the two ends of the scheme.
rows_per_origin = adjustments.groupby("origin_id").size()
washington = sorted(rows_per_origin[rows_per_origin
                                    == rows_per_origin.min()].index)
centres = sorted(rows_per_origin[rows_per_origin
                                 == rows_per_origin.max()].index)
named = sorted(set(adjustments["origin_id"]) | set(adjustments["destination_id"]))

print(f"Zones:                  {len(zone_ids)}")
print(f"Cost matrix:            {base_cost.shape[0]} by {base_cost.shape[1]}"
      f"   ({base_cost.size:,} ordered pairs)")
print(f"Resident workers:       {base_workers.sum():,.0f}")
print(f"Jobs:                   {base_jobs.sum():,.0f}")
print(f"Scheme rows applied:    {len(adjustments)}"
      f"   ({SCHEME_DEFINITION} definition)")
print(f"Zones named:            {len(named)}"
      f"   ({len(washington)} at the Washington end,"
      f" {len(centres)} regional centres)")
print(f"Beta:                   {BETA} per minute")
print(f"Growth to allocate:     {GROWTH_TOTAL:,} jobs")
print()
print(f"ACCESS_ELASTICITY:      {ACCESS_ELASTICITY}")
print(f"LOOP_ITERATIONS:        {LOOP_ITERATIONS}")

## What is held, and what is let go

The fixed case shares the 14,863 new jobs across the 145 zones in proportion to
the employment each already holds. Nothing about the scheme enters that
allocation. Set the elasticity to zero and your reallocation rule returns
exactly this answer, which matters for a reason beyond tidiness: it is also the
land use the do-minimum has. Holding land use fixed means running the
do-something over the do-minimum's employment map.

The variable case starts from the same base year and goes round the loop as many
times as you allowed. Each iteration recomputes accessibility on the land use the
previous one produced, measures every zone's proportional gain from the scheme,
and re-shares the same 14,863 jobs by your rule. Growth is re-shared each
iteration rather than accumulated, so the quantity being distributed never grows
and the two cases always hold the same number of jobs.

Two counts in this notebook are called iterations and they are not the same
thing. Balancing iterations belong to Furness, and they are arithmetic: rows
scaled to resident workers, columns scaled to jobs, repeated until the worst
margin error falls below 0.01 trips. Loop iterations count the times somebody
decided to let the forecast revise its own inputs. The first stops when the
arithmetic stops. The second stops when a modeller says so, and the output
carries no mark to say what they chose.

In [ ]:
weights_base = np.exp(-BETA * base_cost)
weights_scheme = np.exp(-BETA * scheme_cost)


def reallocate(jobs, elasticity):
    """Share the growth out over land use `jobs`, at the given elasticity."""
    do_minimum = weights_base @ jobs
    do_something = weights_scheme @ jobs
    proportional_gain = (do_something - do_minimum) / do_minimum
    weight = base_jobs * (1.0 + elasticity * proportional_gain)
    return base_jobs + GROWTH_TOTAL * weight / weight.sum()


jobs_fixed = reallocate(base_jobs, 0.0)

jobs_variable = base_jobs.copy()
previous = jobs_fixed
trajectory = []
for step in range(1, LOOP_ITERATIONS + 1):
    jobs_variable = reallocate(jobs_variable, ACCESS_ELASTICITY)
    movement = jobs_variable - previous
    biggest = int(np.abs(movement).argmax())
    trajectory.append({
        "iteration": step,
        "measured_against": "fixed case" if step == 1 else f"iteration {step - 1}",
        "jobs_moved": round(float(np.clip(movement, 0.0, None).sum()), 3),
        "largest_single_move": round(float(movement[biggest]), 3),
        "in_zone": zone_names[biggest],
    })
    previous = jobs_variable.copy()

print("THE TWO CASES")
print("-" * 78)
print(f"Fixed     {GROWTH_TOTAL:,} jobs shared in proportion to existing jobs,"
      f" no loop")
print(f"Variable  the same {GROWTH_TOTAL:,} jobs, elasticity"
      f" {ACCESS_ELASTICITY}, {LOOP_ITERATIONS} loop iterations")
print(f"Jobs held after growth:  {jobs_fixed.sum():,.0f} fixed,"
      f" {jobs_variable.sum():,.0f} variable")
print()
print("WHAT EACH LOOP ITERATION MOVED")
print("-" * 78)
print(pd.DataFrame(trajectory).to_string(index=False))

## Re-running the distribution

Both cases are balanced over the scheme cost matrix, with resident workers grown
in the same proportion as jobs and shared out as people already live. Because the
origin margins are then identical across the two runs, whatever separates the
matrices arrived from the destination side, where the elasticity acted.

One consequence is worth seeing once rather than assuming. A doubly constrained
model distributes exactly the trips its margins give it, so the two forecasts
cannot disagree about how many trips there are, in total or in any single zone.
Anyone expecting a land-use response to generate extra travel here will not find
it, and the reason is structural rather than empirical.

In [ ]:
def balance(cost, origins, destinations):
    deterrence = np.exp(-BETA * cost)
    row_factor = np.ones(len(origins))
    col_factor = np.ones(len(destinations))
    for iteration in range(1, MAX_ITERATIONS + 1):
        row_factor = 1.0 / (deterrence * (col_factor * destinations)).sum(axis=1)
        col_factor = 1.0 / (deterrence
                            * (row_factor * origins)[:, None]).sum(axis=0)
        modelled = ((row_factor * origins)[:, None]
                    * (col_factor * destinations)[None, :]
                    * deterrence)
        row_error = np.abs(modelled.sum(axis=1) - origins).max()
        col_error = np.abs(modelled.sum(axis=0) - destinations).max()
        if max(row_error, col_error) < TOLERANCE:
            break
    return modelled, iteration, row_error, col_error


growth_factor = jobs_fixed.sum() / base_workers.sum()
workers = base_workers * growth_factor

matrix_fixed, balance_fixed, row_err_fixed, col_err_fixed = balance(
    scheme_cost, workers, jobs_fixed)
matrix_variable, balance_variable, row_err_var, col_err_var = balance(
    scheme_cost, workers, jobs_variable)

mean_length_fixed = (matrix_fixed * distance).sum() / matrix_fixed.sum()
mean_length_variable = (matrix_variable * distance).sum() / matrix_variable.sum()
worst_row = np.abs(matrix_variable.sum(axis=1) - matrix_fixed.sum(axis=1)).max()

print("BALANCING")
print("-" * 78)
print(f"Tolerance: a margin within {TOLERANCE} trips of its target counts as met")
print(f"Resident workers grown by {growth_factor:.4f}"
      f" to {workers.sum():,.0f}, matching jobs in both cases")
print(f"Fixed     balancing iterations {balance_fixed:3d}"
      f"   worst row error {row_err_fixed:.4f} trips"
      f"   worst column error {col_err_fixed:.4f}")
print(f"Variable  balancing iterations {balance_variable:3d}"
      f"   worst row error {row_err_var:.4f} trips"
      f"   worst column error {col_err_var:.4f}")
print(f"Loop iterations: 0 fixed, {LOOP_ITERATIONS} variable")
print()
print("WHAT DID NOT MOVE")
print("-" * 78)
print(f"Trips distributed:              {matrix_fixed.sum():,.0f} fixed,"
      f" {matrix_variable.sum():,.0f} variable")
print(f"Largest difference in any zone's outbound trips: {worst_row:.6f}")
print(f"Mean trip length:               {mean_length_fixed:.4f} km fixed,"
      f" {mean_length_variable:.4f} km variable")
print(f"  difference: {1000 * (mean_length_variable - mean_length_fixed):+.1f}"
      f" metres, which is below anything this model can resolve")

## The comparison

Seven measures. Three of them describe the region under each assumption and four
describe the scheme, and it is the second group an appraisal would ask you for.

Those four are stated as a percentage of each zone's do-minimum accessibility,
which cancels the growth out of both sides and leaves the two cases comparable.
Read the last two rows before the others. A count is harder to present
selectively than a mean is.

In [ ]:
access_dominimum = weights_base @ jobs_fixed
access_fixed = weights_scheme @ jobs_fixed
access_variable = weights_scheme @ jobs_variable

benefit_fixed = 100.0 * (access_fixed - access_dominimum) / access_dominimum
benefit_variable = (100.0 * (access_variable - access_dominimum)
                    / access_dominimum)

surface = pd.DataFrame({
    "zone_id": zone_ids,
    "zone_name": zone_names,
    "local_authority": authorities,
    "jobs_fixed": jobs_fixed,
    "jobs_variable": jobs_variable,
    "access_dominimum": access_dominimum,
    "access_fixed": access_fixed,
    "access_variable": access_variable,
    "benefit_fixed_pct": benefit_fixed,
    "benefit_variable_pct": benefit_variable,
})
surface["benefit_shift_pp"] = (surface["benefit_variable_pct"]
                               - surface["benefit_fixed_pct"])
surface["jobs_difference"] = surface["jobs_variable"] - surface["jobs_fixed"]

cost_fixed = (matrix_fixed * scheme_cost).sum()
cost_variable = (matrix_variable * scheme_cost).sum()

rows = [
    ("Mean generalised cost per trip, minutes",
     f"{cost_fixed / matrix_fixed.sum():.4f}",
     f"{cost_variable / matrix_variable.sum():.4f}",
     f"{cost_variable / matrix_variable.sum() - cost_fixed / matrix_fixed.sum():+.4f}"),
    ("Total generalised trip-minutes",
     f"{cost_fixed:,.0f}", f"{cost_variable:,.0f}",
     f"{cost_variable - cost_fixed:+,.0f}"),
    ("Mean zonal accessibility",
     f"{access_fixed.mean():,.2f}", f"{access_variable.mean():,.2f}",
     f"{access_variable.mean() - access_fixed.mean():+,.2f}"),
    ("Mean scheme benefit, per cent",
     f"{benefit_fixed.mean():.4f}", f"{benefit_variable.mean():.4f}",
     f"{benefit_variable.mean() - benefit_fixed.mean():+.4f}"),
    ("Median scheme benefit, per cent",
     f"{np.median(benefit_fixed):.4f}", f"{np.median(benefit_variable):.4f}",
     f"{np.median(benefit_variable) - np.median(benefit_fixed):+.4f}"),
    ("Zones with any benefit from the scheme",
     f"{int((benefit_fixed > 1e-4).sum())}",
     f"{int((benefit_variable > 1e-4).sum())}",
     f"{int((benefit_variable > 1e-4).sum()) - int((benefit_fixed > 1e-4).sum()):+d}"),
    ("Zones below their do-minimum accessibility",
     f"{int((benefit_fixed < -1e-4).sum())}",
     f"{int((benefit_variable < -1e-4).sum())}",
     f"{int((benefit_variable < -1e-4).sum()) - int((benefit_fixed < -1e-4).sum()):+d}"),
]

comparison = pd.DataFrame(rows, columns=["measure", "fixed", "variable",
                                         "difference"])

print("FIXED AGAINST VARIABLE LAND USE")
print("-" * 78)
print(comparison.to_string(index=False))
print()

washington_rows = surface["zone_id"].isin(washington)
unnamed_rows = ~surface["zone_id"].isin(named)
region_error = 100.0 * (benefit_variable.mean() - benefit_fixed.mean()) / benefit_fixed.mean()
washington_error = 100.0 * (
    surface.loc[washington_rows, "benefit_variable_pct"].mean()
    - surface.loc[washington_rows, "benefit_fixed_pct"].mean()
) / surface.loc[washington_rows, "benefit_fixed_pct"].mean()

print("HOW WRONG THE FIXED FORECAST IS, AS A PROPORTION OF ITSELF")
print("-" * 78)
print(f"Across all {len(zone_ids)} zones          "
      f"understated by {region_error:.3f} per cent")
print(f"Across the {int(washington_rows.sum())} Washington zones      "
      f"understated by {washington_error:.2f} per cent")
print(f"Across the {int(unnamed_rows.sum())} zones the scheme file never names:"
      f" no proportion exists.")
print(f"  The fixed forecast reports "
      f"{surface.loc[unnamed_rows, 'benefit_fixed_pct'].max():.4f} for every one"
      f" of them, while the")
print(f"  variable forecast runs from "
      f"{surface.loc[unnamed_rows, 'benefit_variable_pct'].min():+.3f} to "
      f"{surface.loc[unnamed_rows, 'benefit_variable_pct'].max():+.3f} per cent.")

## Who gains, and who loses

Under fixed land use the question has a short answer. Ten zones gain, the other
135 change by exactly nothing, and no zone in Tyne and Wear ends below where it
started. That is not the scheme being harmless. It is the scheme file deciding
which rows of the cost matrix were allowed to change, and a zone whose row did
not change cannot move.

Let land use respond and the answer stops being short. Zones the scheme file
never mentions now gain or lose accessibility according to whether employment
moved towards them or away, and the pattern that produces does not respect
local authority boundaries in the way a scheme map implies.

In [ ]:
print("THE TEN ZONES THE SCHEME FILE NAMES")
print("-" * 78)
print(surface[surface["zone_id"].isin(named)]
      .sort_values("benefit_fixed_pct", ascending=False)
      [["zone_id", "zone_name", "benefit_fixed_pct", "benefit_variable_pct",
        "benefit_shift_pp"]]
      .round(3).to_string(index=False))
print()

by_authority = pd.DataFrame({
    "zones": surface.groupby("local_authority").size(),
    "gaining": (surface.assign(g=surface["benefit_variable_pct"] > 1e-4)
                .groupby("local_authority")["g"].sum().astype(int)),
    "losing": (surface.assign(l=surface["benefit_variable_pct"] < -1e-4)
               .groupby("local_authority")["l"].sum().astype(int)),
}).sort_values("gaining", ascending=False)

print("UNDER VARIABLE LAND USE, BY LOCAL AUTHORITY")
print("-" * 78)
print(by_authority.to_string())
print()
print("Under fixed land use, every one of these authorities has 0 losing zones")
print("and between 0 and 7 gaining.")
print()

unnamed = surface[~surface["zone_id"].isin(named)]
print("LARGEST MOVERS AMONG THE 135 ZONES THE SCHEME FILE NEVER NAMES")
print("-" * 78)
print(pd.concat([unnamed.nlargest(5, "benefit_variable_pct"),
                 unnamed.nsmallest(5, "benefit_variable_pct")])
      [["zone_id", "zone_name", "local_authority", "benefit_variable_pct"]]
      .round(3).to_string(index=False))
print()

print("AND WHERE THE JOBS THEMSELVES WENT")
print("-" * 78)
print(f"Zones receiving more than a flat share: "
      f"{int((surface['jobs_difference'] > 1e-6).sum())}")
print(f"Zones receiving less:                   "
      f"{int((surface['jobs_difference'] < -1e-6).sum())}")
print(f"Jobs moved by the elasticity in total:  "
      f"{surface['jobs_difference'].clip(lower=0).sum():,.1f}"
      f" of {GROWTH_TOTAL:,}")
print()
print(pd.concat([surface.nlargest(4, "jobs_difference"),
                 surface.nsmallest(3, "jobs_difference")])
      [["zone_id", "zone_name", "jobs_fixed", "jobs_variable",
        "jobs_difference"]]
      .round(1).to_string(index=False))

## What stands in for network loading

There is no assignment step here and no network in the sense that word usually
carries, so nothing in this notebook can tell you what either forecast does to a
link. But it can report where the trips went. On the pairs the scheme improves
that turns out to be the more interesting question anyway, and the four totals
below are the closest this model comes to a loading comparison.

In [ ]:
scheme_pairs = np.zeros_like(base_cost, dtype=bool)
for origin, destination, change in adjustments.itertuples(index=False):
    scheme_pairs[position[origin], position[destination]] = True

wash_index = [position[z] for z in washington]
centre_index = [position[z] for z in centres]


def compare(label, selector):
    before = selector(matrix_fixed)
    after = selector(matrix_variable)
    print(f"  {label:44s} {before:9,.1f} {after:9,.1f} "
          f"{after - before:+8.1f}   {100 * (after - before) / before:+.3f}"
          f" per cent")


print("TRIPS, FIXED THEN VARIABLE")
print("-" * 78)
compare(f"On the {len(adjustments)} pairs the scheme improves",
        lambda m: m[scheme_pairs].sum())
compare("Beginning in Washington",
        lambda m: m[wash_index, :].sum())
compare("Washington to the three regional centres",
        lambda m: m[np.ix_(wash_index, centre_index)].sum())
compare("Washington to Washington",
        lambda m: m[np.ix_(wash_index, wash_index)].sum())
print()

difference = matrix_variable - matrix_fixed
largest = np.argsort(-np.abs(difference), axis=None)[:8]
print("THE EIGHT ORIGIN-DESTINATION PAIRS THAT MOVED MOST")
print("-" * 78)
for flat in largest:
    i, j = np.unravel_index(flat, difference.shape)
    print(f"  {zone_names[i][:30]:30s} to {zone_names[j][:30]:30s}"
          f" {matrix_fixed[i, j]:7.1f} {matrix_variable[i, j]:7.1f}"
          f" {difference[i, j]:+6.1f}")

## The two forecasts, zone by zone

The left-hand panel puts each zone's scheme benefit under fixed land use against
the same zone's benefit under your variable run, with the line every point would
sit on if the assumption made no difference. The right-hand panel takes the 135
zones the scheme file never names and shows the distribution of what the
variable case gives them, against the single value the fixed case gives them all.

In [ ]:
import matplotlib.pyplot as plt

RED = "#C8102E"
GREEN = "#025944"
GREY = "#9CA3AF"

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))

limit = max(surface["benefit_fixed_pct"].max(),
            surface["benefit_variable_pct"].max()) * 1.08
axes[0].plot([-2, limit], [-2, limit], color=GREY, linewidth=1)
axes[0].scatter(surface["benefit_fixed_pct"], surface["benefit_variable_pct"],
                s=26, color=RED, alpha=0.75, edgecolor="none")
axes[0].set_xlim(-3, limit)
axes[0].set_ylim(-3, limit)
axes[0].set_xlabel("Scheme benefit, land use fixed (per cent)")
axes[0].set_ylabel("Scheme benefit, land use variable (per cent)")
axes[0].set_title(f"All {len(surface)} zones")
axes[0].annotate(f"{len(unnamed)} zones sit here",
                 xy=(0.6, 0.0), xytext=(11.0, 3.0), fontsize=9, color=GREY,
                 arrowprops=dict(arrowstyle="-", color=GREY, linewidth=0.8))

axes[1].hist(unnamed["benefit_variable_pct"], bins=24, color=GREEN,
             edgecolor="white", linewidth=0.5)
axes[1].axvline(0.0, color=RED, linewidth=1.4)
axes[1].set_xlabel("Scheme benefit, land use variable (per cent)")
axes[1].set_ylabel("Zones")
axes[1].set_title(f"The {len(unnamed)} zones the scheme file never names")

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(f"Elasticity {ACCESS_ELASTICITY}, {LOOP_ITERATIONS}"
             f" loop iterations")
plt.tight_layout()
plt.show()

print("The red line on the right-hand panel is the fixed forecast's answer for")
print("all 135 of those zones.")

## The export

One row per zone, both cases side by side, with your two parameter values
carried on every row. The tabulation you are asked to submit is easier in a
spreadsheet than on this page, and the parameters travel with the file because a
column of accessibility changes with no elasticity attached cannot be read by
anybody, including you, a fortnight from now.

In [ ]:
export = surface.copy()
export.insert(3, "access_elasticity", ACCESS_ELASTICITY)
export.insert(4, "loop_iterations", LOOP_ITERATIONS)

EXPORT_FILE = "fixed-vs-variable-by-zone.csv"
export.round(4).to_csv(EXPORT_FILE, index=False, encoding="utf-8-sig")

print(f"Written: {EXPORT_FILE}")
print(f"Rows: {len(export)}   Columns: {len(export.columns)}")
print()
print("Columns:")
for column in export.columns:
    print(f"  {column}")
print()
print("Right-click the file in the file browser and choose Download.")
print("Browser storage is not permanent.")

## What this comparison does not settle

Four things.

The growth is a fixed quantity, decided outside the model and unaffected by the
scheme. All your elasticity does is decide where it lands, so what the variable
case represents is displacement rather than growth: every job it draws to
Springwell & Usworth is a job that did not go to Dunston & Teams, and the gains
and the falls cancel exactly when you add them up.

Which constrains what you can write about the zones that fell. They have not
been harmed by a Metro extension. They have received a smaller share of an
increase that was arriving somewhere regardless, and a sentence saying the
scheme makes North Tyneside worse off would be claiming something this model
cannot support. That closure is not a quirk of the exercise, incidentally.
DELTA and TELMoS also take regional control totals from outside and distribute
them within, and the commuting data underneath this model excludes every journey
crossing the study area boundary, so Tyne and Wear has no way of attracting
growth from beyond it here.

Divergence between the two forecasts is usually argued to widen over a
forecast horizon, on the reasoning that development takes years to respond and a
twenty-year appraisal gives it time to. That argument is not tested here and
cannot be. There is no time in this model: no year in which anything happens, no
lag between an accessibility change and a building, and no way to distinguish an
iteration from a decade. Treat the horizon claim as an argument you have reasons
to find plausible, and keep it separate from the numbers above, which are
results.

Then the constraint on every figure printed above. The variable forecast is not
the better number. It rests on an elasticity you chose rather than estimated,
run for a count of iterations you also chose, and moving either one moves the
answer without making the output look any less certain. Which is why both belong
in your written answer, beside the figures they produced.